# ADR Security Validator - AMD Developer Cloud Deployment Guide

This notebook guides you through the deployment of the ADR Security Validator on an AMD Instinct™ GPU (like MI300X or MI210) using the ROCm™ software stack.

### ⚠️ Troubleshooting: "getcwd: cannot access parent directories"
If you see this error, it means the directory you were in was deleted. Run the cell below to fix the working directory.

In [ ]:
import os
project_dir = "/root/AMD-Developer-Hackathon"
if os.path.exists(project_dir):
    %cd {project_dir}
print(f"Working directory set to: {os.getcwd()}")

## 1. Verify GPU Status

In [ ]:
!rocm-smi

## 2. Environment Setup
Install the optimized version of PyTorch for ROCm and other project dependencies.

In [ ]:
# Install PyTorch optimized for ROCm 6.2
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/rocm6.2

# Install project requirements
!pip install -r backend/requirements.txt

# Install optimized training library Unsloth for ROCm
!pip install "unsloth[rocm] @ git+https://github.com/unslothai/unsloth.git"

## 3. Vector Database (Qdrant)
Start the Qdrant vector database using Docker. This will store our historical ADRs.

In [ ]:
!docker run -d -p 6333:6333 -p 6334:6334 qdrant/qdrant

## 4. Data Indexing
Index the ADR data from Kubernetes, Django, and Rust into Qdrant.

In [ ]:
%cd backend
!python qdrant_setup.py --embed
%cd ..

## 5. Fine-tuning (Optional)
Run the fine-tuning script to optimize Qwen3-8B for ADR validation and security analysis. On an MI300X, this takes about 1.5 hours.

In [ ]:
%cd fine-tuning
!python fine_tune_qwen3_8b.py
%cd ..

## 6. Start the API Backend
Launch the FastAPI server. 

**Note:** To keep the server running, it's recommended to run this in a terminal or use a tool like `nohup` or `screen`. 

```bash
cd backend
uvicorn main:app --host 0.0.0.0 --port 8000
```

In [ ]:
# This will block the notebook if run directly.
# !cd backend && uvicorn main:app --host 0.0.0.0 --port 8000